<a href="https://colab.research.google.com/github/OutOfMystic/tetris-openenv/blob/main/tetris_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tetris OpenEnv - Training with Unsloth + GRPO

Train an LLM agent to play Tetris using GRPO (Group Relative Policy Optimization).

**Environment**: Tetris on HF Spaces via OpenEnv 0.2.1
**Model**: Qwen2.5-7B-Instruct (4-bit via Unsloth)
**Training**: GRPO via TRL

Runtime: **A100 GPU** (40GB VRAM, bf16, Colab Pro)

In [1]:
# Cell 1: Install dependencies
!pip install unsloth trl openenv-core datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.7/633.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Cell 2: Load model with Unsloth (4-bit quantization + LoRA for A100)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,                # A100 has headroom — r=32 for richer adaptation
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

In [3]:
# Cell 3: Define Tetris client (standalone, no local package needed)
import json
import websockets.sync.client as ws_client

TETRIS_URL = "wss://vortexedsquirrel-tetris-env.hf.space/ws"

class TetrisClient:
    """Lightweight Tetris env client for Colab."""
    def __init__(self, url=TETRIS_URL):
        self.url = url
        self.ws = None

    def connect(self):
        self.ws = ws_client.connect(self.url, open_timeout=30)
        return self

    def _send_recv(self, msg):
        self.ws.send(json.dumps(msg))
        return json.loads(self.ws.recv(timeout=30))

    def reset(self, seed=None):
        data = {"seed": seed} if seed else {}
        resp = self._send_recv({"type": "reset", "data": data})
        d = resp["data"]
        obs = d.get("observation", d)
        return {"observation": obs, "reward": d.get("reward", 0), "done": d.get("done", False)}

    def step(self, action_str):
        resp = self._send_recv({"type": "step", "data": {"action": action_str, "metadata": {}}})
        d = resp["data"]
        obs = d.get("observation", d)
        return {"observation": obs, "reward": d.get("reward", 0), "done": d.get("done", False)}

    def close(self):
        if self.ws:
            self.ws.close()

# Quick test
client = TetrisClient().connect()
r = client.reset(seed=42)
print(f"Connected! Piece: {r['observation']['current_piece']}")
r = client.step("drop")
print(f"Step OK. Reward: {r['reward']}, Done: {r['done']}")
client.close()
print("Environment connection verified!")

Connected! Piece: L
Step OK. Reward: -7, Done: False
Environment connection verified!


In [ ]:
# Cell 4: Generate training prompts — SEQUENCE-BASED approach
# Model outputs a sequence of actions, we evaluate full game performance
import random

# Download game engine to run locally (no network calls needed)
!wget -q -O game_engine.py https://raw.githubusercontent.com/OutOfMystic/tetris-openenv/main/src/tetris_env/server/game_engine.py

from game_engine import TetrisEnv

SYSTEM_PROMPT = """You are a Tetris AI. You see a board and must output a SEQUENCE of actions to play.
Valid actions: left, right, rotate_cw, rotate_ccw, drop, down
Output actions separated by spaces. Example: left left rotate_cw drop right drop
Strategy: position pieces to fill rows completely. Clearing multiple lines at once = bonus.
Output ONLY the action sequence, nothing else."""

VALID_ACTIONS = ["left", "right", "rotate_cw", "rotate_ccw", "drop", "down"]

def make_prompt(result):
    return f"""Board:
{result['board']}

Current piece: {result['current_piece']} (shape: {result['current_piece_shape']})
Next piece: {result['next_piece']}
Score: {result['score']} | Lines: {result['total_lines']} | Height: {result['max_height']} | Holes: {result['holes']}

Output your action sequence:"""

def generate_training_prompts(n_prompts=400, max_random_steps=30):
    prompts = []
    for i in range(n_prompts):
        env = TetrisEnv(seed=i)
        result = env.reset(seed=i)
        # Play some random steps to get varied board states
        n_steps = random.randint(0, max_random_steps)
        for _ in range(n_steps):
            action = random.choice(VALID_ACTIONS + ["drop"])
            result = env.step(action)
            if result['done']:
                break
        if not result['done']:
            prompt = make_prompt(result)
            prompts.append({
                "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": prompt}],
            })
    print(f"Generated {len(prompts)} training prompts")
    return prompts

train_prompts = generate_training_prompts(400)

In [7]:
# Cell 5: Create HF Dataset from prompts
from datasets import Dataset

dataset = Dataset.from_list(train_prompts)
print(f"Dataset size: {len(dataset)}")
print(f"Example prompt (user message):")
print(dataset[0]['prompt'][1]['content'][:300])

Dataset size: 199
Example prompt (user message):
Board:
+----------+
|..........|
|..........|
|..........|
|...@@@@...|
|..........|
|..........|
|..........|
|..........|
|...###....|
|.....#....|
|.....#....|
|.....#....|
|....##....|
|.....#....|
|.....#....|
|....##....|
|....#.....|
|...##.....|
|...###....|
|....##....|
+----------+

Curren


In [ ]:
# Cell 6: Reward function — plays full action sequence on local engine
# Game over = losing individual (heavy penalty)
# More lines + longer survival = better individual

def tetris_reward_func(completions, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]['content'].strip().lower() if isinstance(completion, list) else str(completion).strip().lower()

        # Parse action sequence from model output
        tokens = text.replace(",", " ").replace("\n", " ").split()
        actions = [t for t in tokens if t in VALID_ACTIONS]

        if len(actions) == 0:
            rewards.append(-20.0)  # no valid actions = very bad
            continue

        # Play full sequence on fresh engine (seed=i for reproducibility)
        env = TetrisEnv(seed=i)
        env.reset(seed=i)

        total_lines = 0
        steps_played = 0
        game_over = False

        for action in actions:
            result = env.step(action)
            steps_played += 1
            total_lines = result['total_lines']
            if result['done']:
                game_over = True
                break

        # Reward structure:
        # - Lines cleared are the main reward (with combo bonus from engine)
        # - Surviving longer = small bonus
        # - Game over = losing individual (heavy penalty)
        # - Clean board (low holes, low height) = bonus
        reward = 0.0
        reward += total_lines * 200.0          # main goal: clear lines
        reward += steps_played * 0.5           # survive longer = better
        reward -= result['holes'] * 2.0        # penalize messy boards
        reward -= result['max_height'] * 1.0   # penalize tall stacks
        if game_over:
            reward -= 50.0                     # game over = bad individual
        if len(actions) < 3:
            reward -= 10.0                     # too short sequences are lazy

        rewards.append(reward)

    return rewards

# Test with sample completions
test_completions = [
    [{"content": "left left rotate_cw drop right right drop left drop"}],
    [{"content": "drop drop drop drop drop drop drop drop"}],
    [{"content": "this is not a valid action at all"}],
    [{"content": "left"}],
]
test_rewards = tetris_reward_func(test_completions)
print(f"Test rewards: {test_rewards}")
print("Good: varied rewards → GRPO can learn from differences!")

In [ ]:
# Cell 7: Configure and run GRPO training (A100 optimized)
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="tetris-agent-grpo",
    num_train_epochs=3,
    per_device_train_batch_size=4,    # A100 handles bigger batches
    gradient_accumulation_steps=2,    # effective batch = 4*2 = 8
    num_generations=8,                # 8 responses per prompt — richer GRPO signal
    max_completion_length=256,        # longer sequences = more gameplay per generation
    max_prompt_length=512,
    learning_rate=5e-6,
    logging_steps=1,
    save_steps=50,
    bf16=True,                        # A100 supports bf16 natively
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[tetris_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 8: Plot reward curve
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_logs = [l for l in logs if 'loss' in l]
reward_logs = [l for l in logs if 'reward' in l or 'rewards/mean' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_logs:
    axes[0].plot([l.get('step', i) for i, l in enumerate(train_logs)],
                 [l['loss'] for l in train_logs])
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')

reward_key = 'reward' if reward_logs and 'reward' in reward_logs[0] else 'rewards/mean'
if reward_logs:
    axes[1].plot([l.get('step', i) for i, l in enumerate(reward_logs)],
                 [l.get(reward_key, 0) for l in reward_logs])
    axes[1].set_title('Mean Reward (should go up!)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Reward')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print("Reward curve saved to reward_curve.png")

In [ ]:
# Cell 9: Test trained agent (sequence-based)
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

client = TetrisClient().connect()
result = client.reset(seed=123)

total_reward = 0
games_played = 0
max_games = 3  # play 3 games to see consistency

for game in range(max_games):
    result = client.reset(seed=123 + game)
    game_reward = 0
    step_num = 0

    while not result['done'] and step_num < 200:
        prompt_text = make_prompt(result['observation'])
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text}
        ]
        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
        outputs = model.generate(inputs, max_new_tokens=256, temperature=0.3)
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip().lower()

        # Parse action sequence and execute
        tokens = response.replace(",", " ").replace("\n", " ").split()
        actions = [t for t in tokens if t in VALID_ACTIONS]

        if not actions:
            actions = ["drop"]

        for action in actions:
            result = client.step(action)
            game_reward += result['reward']
            step_num += 1
            if result['done']:
                break
        if result['done']:
            break

    print(f"Game {game+1}: steps={step_num}, score={result['observation']['score']}, "
          f"lines={result['observation']['total_lines']}, reward={game_reward:.1f}")
    total_reward += game_reward

print(f"\nAvg reward over {max_games} games: {total_reward/max_games:.1f}")
print(f"\nFinal board (last game):")
print(result['observation']['board'])
client.close()

In [ ]:
# Cell 10: Push trained model to HF Hub
model.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
tokenizer.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
print("Model pushed to https://huggingface.co/VortexedSquirrel/tetris-agent-grpo")